# PhysioGraphSleep — Training & Evaluation Pipeline

Heterojen çizge sinir ağları ile tek/iki kanallı polisomnografi epok
sınıflandırma deneylerini yürüten uçtan uca not defteri. Hedef protokol:
**Sleep-EDF-20 / Sleep-EDF-78**, AASM 5-sınıf (W / N1 / N2 / N3 / REM).

**Bölüm planı**

| § | İçerik |
|---|---|
| 1 | Ortam kurulumu, paket yüklemesi, GPU kontrolü |
| 2 | Deney yapılandırması ve ablation anahtarları |
| 3 | Veri yükleme ve sınıf dağılımı özeti |
| 4 | Dataset / DataLoader inşası |
| 5 | Model yapılandırması ve sanity forward |
| 6 | Eğitim döngüsü, canlı grafikler, WandB logging |
| 7 | Test değerlendirmesi, post-processing, kalibrasyon |
| 8 | Bileşen tanılama (aktivasyon, lineer prob, modalite, füzyon) |
| 9 | Ablation sonuçlarının toplulaştırılması |
| 10 | K-fold subject-wise cross-validation (Sleep-EDF-20 & 78) |

Not: Tüm çıktılar `physiographsleep/outputs/`, checkpoint'ler
`physiographsleep/checkpoints/` altına yazılır. WandB opsiyoneldir
(`WANDB_ENABLED=True` ile aktive edilir).

## 1. Ortam Kurulumu

Depo yolu çözümü (Colab → git clone, yerel → mevcut çalışma dizini),
bağımlılık yüklemesi, donanım kontrolü. Yapılandırma sabitleri
yalnızca §2'de tanımlanır; bu bölümdeki hücreler tekrar çalıştırıldığında
eğitim durumuna etki etmez.

In [ ]:
"""Runtime warnings and MNE logging suppression."""
import os
import sys
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Channels contain different.*")
warnings.filterwarnings("ignore", message=".*Highpass cutoff frequency.*")
warnings.filterwarnings("ignore", message=".*Lowpass cutoff frequency.*")
os.environ["MNE_LOGGING_LEVEL"] = "ERROR"

REPO_URL = "https://github.com/0nur0duncu/physiographsleep.git"
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab: shared memory sınırını DataLoader worker'ları için genişlet
    os.system("sudo mount -o remount,size=2G /dev/shm 2>/dev/null")
    PROJECT_DIR = "/content/physiographsleep"
    if os.path.exists(PROJECT_DIR):
        os.system(f"cd {PROJECT_DIR} && git pull --quiet")
    else:
        os.system(f"git clone --quiet {REPO_URL} {PROJECT_DIR}")
    PARENT_DIR = "/content"
else:
    # Yerel: notebook physiographsleep/ içinde çalışır; parent'ı PYTHONPATH'e ekle.
    cwd = os.path.abspath(".")
    PROJECT_DIR = cwd if os.path.basename(cwd) == "physiographsleep" else os.path.dirname(cwd)
    PARENT_DIR = os.path.dirname(PROJECT_DIR)

if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)
os.chdir(PARENT_DIR)
print(f"Runtime  : {'Colab' if IS_COLAB else 'Local'}")
print(f"Project  : {PROJECT_DIR}")
print(f"Workdir  : {os.getcwd()}")

In [ ]:
"""Bağımlılıkları yükle (Colab için zorunlu, yerel için pip verify)."""
if IS_COLAB:
    os.system(f"pip install -q -r {PROJECT_DIR}/requirements.txt")
    os.system("pip install -q wandb")

In [ ]:
"""Donanım envanteri."""
import numpy as np
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"GPU      : {dev.name}")
    print(f"VRAM     : {dev.total_memory / 1024**3:.1f} GB")

In [ ]:
"""1.3 — Ortam tanılaması.

CPU/GPU dengesizliği, bfloat16 desteği ve DataLoader worker sayısını
doğrular. Colab T4 + Linux için beklenen tipik değerler:
    cuda.is_available = True, bf16 = True, num_workers ≥ 2, pin_memory = True.
Bunlardan biri False ise eğitim süresi 2-10 kat artabilir.
"""
import platform
from physiographsleep.configs import ExperimentConfig

_cfg = ExperimentConfig()
print(f"{'torch':<18}: {torch.__version__}")
print(f"{'platform':<18}: {platform.system()} ({platform.machine()})")
print(f"{'cuda.available':<18}: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{'cuda.device':<18}: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"{'cuda.vram_gb':<18}: {props.total_memory / 1024**3:.1f}")
    print(f"{'bf16_supported':<18}: {torch.cuda.is_bf16_supported()}")
    print(f"{'cudnn.benchmark':<18}: True (set by trainer)")
print(f"{'num_workers':<18}: {_cfg.data.num_workers}"
      f"{'  ⚠ Linux/Colab icin 2+ olmali' if _cfg.data.num_workers == 0 and not platform.system().startswith('Win') else ''}")
print(f"{'batch_size':<18}: {_cfg.data.batch_size}")
print(f"{'pin_memory':<18}: {_cfg.data.pin_memory}")
print(f"{'seq_len':<18}: {_cfg.data.seq_len}")

## 2. Deney Yapılandırması

`ExperimentConfig` varsayılanları üzerine uygulanan yol, kanal modu,
ablation anahtarı ve kayıp fonksiyonu ağırlık stratejisi. Tüm çıktılar
`PROJECT_DIR` altında kalır; harici (ör. Drive) yola yazılmaz.

In [ ]:
"""Experiment configuration — paths, channel mode, logging, scheduler."""
from pathlib import Path

from physiographsleep.configs import ExperimentConfig, sync_channel_config
from physiographsleep.utils.reproducibility import get_device, set_seed

PROJECT_DIR = Path(PROJECT_DIR)
assert (PROJECT_DIR / "__init__.py").exists(), (
    f"physiographsleep package not found at {PROJECT_DIR}"
)

# --- Runtime toggles -----------------------------------------------------
USE_EOG            = False   # 2-kanal (EEG + EOG horizontal) için True
USE_TORCH_COMPILE  = False   # ilk epoch +60 s, sonraki epoch'lar ~1.5× hızlı
WANDB_ENABLED      = True
WANDB_PROJECT      = "neurographTdraft"
WANDB_RUN_NAME     = "physiographsleep_sota_revert"

# --- Output paths --------------------------------------------------------
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

config = ExperimentConfig()
if IS_COLAB:
    config.data.data_dir  = "/content/data/sleepedf"
    config.data.cache_dir = "/content/cache"
else:
    config.data.data_dir  = str(PROJECT_DIR / "data" / "sleepedf20")
    config.data.cache_dir = str(PROJECT_DIR / "cache")
config.train.checkpoint_dir = str(PROJECT_DIR / "checkpoints")
config.train.log_dir        = str(PROJECT_DIR / "logs")

# --- Channel mode (waveform + spectral feature-dim sync) -----------------
if USE_EOG:
    config.data.use_eog = True
sync_channel_config(config)

# --- Training schedule ---------------------------------------------------
# epochs ve cosine t_max eşleşmeli; warmup ve patience ayrı yönetilir.
config.train.epochs           = 30
config.train.scheduler.t_max  = 30
config.train.lr               = 1e-3
config.train.n1_boost         = 1.3

set_seed(config.seed)
device = get_device("auto")

In [ ]:
"""Ablation anahtarları + kayıp fonksiyonu ağırlık stratejisi.

Üç yapısal bileşen bağımsız olarak açılıp kapatılabilir; her kombinasyon
için notebook baştan sona bir kez çalıştırıldığında sonuç otomatik olarak
`outputs/ablation/<ABLATION_NAME>.json` dosyasına yazılır ve §9'daki
toplu tablo bunları karşılaştırır.

Weight strategies:
    none          — düzgün ağırlık (N1 dengesi WeightedRandomSampler'dan gelir)
    inverse_freq  — 1/count (klasik, örnekleyici ile birlikte önerilmez)
    class_balanced— Cui et al. CVPR 2019 effective-number ağırlıkları
    adaptive_f1   — SeriesSleepNet 2023; F1-tabanlı dinamik ağırlık (+1–6 pp N1)
"""
USE_PATHWAY     = False
USE_FUSION      = False
USE_N1_MIXUP    = True
WEIGHT_STRATEGY = "adaptive_f1"

if not USE_PATHWAY:
    config.model.graph.edge_pathways = None
if not USE_FUSION:
    config.model.fusion = None
if not USE_N1_MIXUP:
    config.train.n1_mixup = None
config.train.loss.weight_strategy = WEIGHT_STRATEGY

_WS_SHORT = {"none": "n", "adaptive_f1": "af1", "class_balanced": "cb", "inverse_freq": "if"}
ABLATION_NAME = (
    f"p{int(USE_PATHWAY)}_f{int(USE_FUSION)}_m{int(USE_N1_MIXUP)}"
    f"_w{_WS_SHORT.get(WEIGHT_STRATEGY, WEIGHT_STRATEGY)}"
)


def _fmt_fusion(fusion) -> str:
    if fusion is None:
        return "disabled"
    return f"init_lambda={fusion.init_lambda}, detach_gnn={fusion.detach_gnn_for_lambda}"


def _fmt_mixup(mx) -> str:
    if mx is None:
        return "disabled"
    return f"prob={mx.prob}, alpha={mx.alpha}"


print(f"Ablation : {ABLATION_NAME}")
print(f"Device   : {device}")
print(f"Channel  : {'EEG Fpz-Cz + EOG horizontal (2ch)' if USE_EOG else 'EEG Fpz-Cz (1ch)'}")
print(f"Subjects : {config.data.num_subjects}  "
      f"(train={config.data.train_subjects}, val={config.data.val_subjects})")
print(f"Batch    : {config.data.batch_size}  |  workers={config.data.num_workers}")
print(f"Epochs   : {config.train.epochs}  |  warmup={config.train.scheduler.warmup_epochs}  "
      f"|  patience={config.train.patience}")
print(f"LR       : {config.train.lr}  |  N1 boost={config.train.n1_boost}×")
print(f"Graph    : layers={config.model.graph.num_layers}  "
      f"heads={config.model.graph.num_heads}  drop_path={config.model.graph.drop_path}")
print()
print(f"  pathway       : {'on' if USE_PATHWAY else 'off'}  "
      f"(edge_pathways={config.model.graph.edge_pathways})")
print(f"  λ-fusion      : {'on' if USE_FUSION else 'off'}  "
      f"({_fmt_fusion(config.model.fusion)})")
print(f"  N1-mixup      : {'on' if USE_N1_MIXUP else 'off'}  "
      f"({_fmt_mixup(config.train.n1_mixup)})")
print(f"  weights       : {WEIGHT_STRATEGY}")
if WEIGHT_STRATEGY == "adaptive_f1":
    ls = config.train.loss
    print(f"    warmup={ls.adaptive_warmup}  K={ls.adaptive_K}  γ={ls.adaptive_gamma}")
elif WEIGHT_STRATEGY == "class_balanced":
    print(f"    β={config.train.loss.cb_beta}")
print(f"  label_smooth  : {config.train.loss.label_smoothing}")
print(f"  weight_decay  : {config.train.optimizer.weight_decay}")

In [ ]:
"""Weights & Biases başlatma (opsiyonel).

Yerel makinede ``wandb login`` veya ``WANDB_API_KEY`` ortam değişkeni
gerekir. Colab'da Secrets panelinden ``WANDB_API_KEY`` okunur.
"""
wandb_run = None
if WANDB_ENABLED:
    import wandb

    try:
        from google.colab import userdata  # type: ignore
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass

    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=f"{WANDB_RUN_NAME}_{ABLATION_NAME}",
        dir=str(OUTPUTS_DIR),
        config={
            "num_subjects":    config.data.num_subjects,
            "use_eog":         config.data.use_eog,
            "batch_size":      config.data.batch_size,
            "seq_len":         config.data.seq_len,
            "epochs":          config.train.epochs,
            "lr":              config.train.lr,
            "weight_decay":    config.train.optimizer.weight_decay,
            "label_smoothing": config.train.loss.label_smoothing,
            "n1_boost":        config.train.n1_boost,
            "patience":        config.train.patience,
            "ablation_name":   ABLATION_NAME,
            "weight_strategy": WEIGHT_STRATEGY,
        },
    )
    print(f"WandB    : {wandb_run.url}")
else:
    print("WandB    : disabled")

## 3. Veri Yükleme ve DataLoader İnşası

EDF dosyaları indirilir (yoksa), filtrelenir, 30 s'lik epoklara bölünür,
spektral özellikler önceden hesaplanır ve disk önbelleğine yazılır.
Öznitelik tensörleri `physiographsleep.data.dataset.SleepEDFDataset`
içinde dizisel pencere (`seq_len=25`) olarak paketlenir.

In [ ]:
"""Veri setini yükle ve sınıf dağılımını raporla."""
from physiographsleep.data.loader import load_sleep_edf

STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]

data = load_sleep_edf(config.data)

print(f"{'split':<6} {'epochs':>8} {'shape':<22} {'spectral':<22}")
print("-" * 60)
for split_name, split_data in data.items():
    epochs = split_data["epochs"]
    labels = split_data["labels"]
    spectral = split_data.get("spectral")
    spec_shape = str(spectral.shape) if spectral is not None else "N/A"
    print(f"{split_name:<6} {len(labels):>8d} {str(epochs.shape):<22} {spec_shape:<22}")

train_labels = data["train"]["labels"]
counts = np.bincount(train_labels, minlength=5)
total = len(train_labels)
print(f"\nTrain class distribution ({total:,} epochs):")
for i, name in enumerate(STAGE_NAMES):
    pct = 100 * counts[i] / total
    bar = "█" * int(pct / 2)
    print(f"  {name:<3} {counts[i]:>7,} ({pct:>5.1f}%) {bar}")

In [ ]:
"""Train / val Dataset ve DataLoader nesneleri."""
from torch.utils.data import DataLoader

from physiographsleep.data.dataset import SleepEDFDataset
from physiographsleep.data.transforms import SleepTransforms

train_transform = SleepTransforms(config.data)
train_ds = SleepEDFDataset(
    data["train"]["epochs"],
    data["train"]["labels"],
    config=config.data,
    transform=train_transform,
    spectral=data["train"].get("spectral"),
)
val_ds = SleepEDFDataset(
    data["val"]["epochs"],
    data["val"]["labels"],
    config=config.data,
    spectral=data["val"].get("spectral"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.data.batch_size,
    shuffle=False,
    num_workers=config.data.num_workers,
    pin_memory=True,
    persistent_workers=config.data.num_workers > 0,
    prefetch_factor=4 if config.data.num_workers > 0 else None,
)
print(f"Train : {len(train_ds):>6,} samples")
print(f"Val   : {len(val_ds):>6,} samples ({len(val_loader)} batches)")

## 4. Model İnşası ve Sanity Forward

`PhysioGraphSleep` mimarisinin başlatılması, parametre dağılımının
raporlanması ve tek-batch ileri geçişle tensör boyutlarının doğrulanması.
`USE_TORCH_COMPILE=True` ise eğitim başlamadan önce derleme (ilk adımda
~60 s gecikme) uygulanır.

In [ ]:
"""Model instantiation, parameter inventory, sanity forward."""
from physiographsleep.models.physiographsleep import PhysioGraphSleep

model = PhysioGraphSleep(config.model)
param_counts = model.count_parameters()

print("Parameter counts:")
for name, count in param_counts.items():
    print(f"  {name:<22} {count:>10,}")

# Tek-batch ileri geçiş — şekil doğrulama.
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader))
    sig = batch["signal"][:2]
    spec = batch.get("spectral",
                     torch.zeros(2, config.data.seq_len, 5, 42))[:2]
    out = model(sig, spec)
print(f"\nForward check: signal={tuple(sig.shape)} → stage={tuple(out['stage'].shape)}")

if USE_TORCH_COMPILE and torch.cuda.is_available():
    model = torch.compile(model, mode="reduce-overhead", dynamic=False)
    print("torch.compile applied.")

## 5. Eğitim Döngüsü

`Trainer` her epoch sonunda kayıt defterine satır yazar, canlı 4-panel
grafik çizer ve WandB oturumu etkinse metrikleri uzak servise loglar.
Checkpoint dosyası zaten mevcutsa eğitim atlanıp §6'da post-processing'e
doğrudan geçilebilir.

In [ ]:
"""Kayıp fonksiyonu ve sınıf ağırlığı stratejisinin uygulanması."""
import gc

from physiographsleep.data.spectral import SpectralFeatureExtractor
from physiographsleep.models.losses import (
    MultiTaskLoss,
    compute_class_balanced_weights,
    compute_inverse_freq_weights,
)
from physiographsleep.utils.logging_utils import setup_logger

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory: alloc={alloc:.2f} GB | reserved={reserved:.2f} GB")

train_counts = np.bincount(data["train"]["labels"], minlength=5)
_ws = config.train.loss.weight_strategy
if _ws == "inverse_freq":
    class_weights = torch.from_numpy(compute_inverse_freq_weights(train_counts)).float()
elif _ws == "class_balanced":
    class_weights = torch.from_numpy(
        compute_class_balanced_weights(train_counts, beta=config.train.loss.cb_beta)
    ).float()
elif _ws == "adaptive_f1":
    class_weights = torch.ones(5)
else:
    class_weights = None

loss_fn = MultiTaskLoss(config.train.loss, class_weights=class_weights)
spectral_ext = SpectralFeatureExtractor(config.data)
logger = setup_logger(log_dir=config.train.log_dir)
print(f"Strategy : {_ws}")
print(f"Loss     : FocalLoss(γ={config.train.loss.focal_gamma}, "
      f"ls={config.train.loss.label_smoothing})")

In [ ]:
# ============================================================
# Training history + per-epoch callback
# ============================================================
HISTORY = {"epoch": [],
           "train_loss": [], "val_loss": [],
           "train_acc": [], "train_mf1": [],
           "val_acc": [], "val_mf1": [], "val_mf1_gnn": [], "val_kappa": [],
           "val_f1_W": [], "val_f1_N1": [], "val_f1_N2": [],
           "val_f1_N3": [], "val_f1_REM": []}

HEADER_PRINTED = {"v": False}
def _print_header():
    if HEADER_PRINTED["v"]:
        return
    cols = ("Epoch", "TrLoss", "VlLoss",
            "TrAcc", "TrMF1", "VlAcc", "VlMF1", "GnnMF1", "Kappa",
            "F1_W", "F1_N1", "F1_N2", "F1_N3", "F1_REM")
    print(" | ".join(f"{c:>7s}" for c in cols))
    print("-" * (10 * len(cols)))
    HEADER_PRINTED["v"] = True

def _draw_live_plot():
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    epochs_x = list(range(len(HISTORY["epoch"])))

    axes[0, 0].plot(epochs_x, HISTORY["train_loss"], "-o", ms=3, label="train")
    axes[0, 0].plot(epochs_x, HISTORY["val_loss"],   "-s", ms=3, label="val")
    axes[0, 0].set_title("Loss"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(epochs_x, HISTORY["train_acc"], "-o", ms=3, label="train")
    axes[0, 1].plot(epochs_x, HISTORY["val_acc"],   "-s", ms=3, label="val")
    axes[0, 1].set_title("Accuracy"); axes[0, 1].set_ylim(0, 1); axes[0, 1].grid(alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(epochs_x, HISTORY["train_mf1"], "-o", ms=3, label="train")
    axes[1, 0].plot(epochs_x, HISTORY["val_mf1"],   "-s", ms=3, label="val (fused)")
    # GNN-only MF1 — fusion kapalıyken val_mf1 ile aynı; açıkken ayrık eğri
    if any(v is not None for v in HISTORY["val_mf1_gnn"]):
        gnn_y = [v if v is not None else float("nan") for v in HISTORY["val_mf1_gnn"]]
        axes[1, 0].plot(epochs_x, gnn_y, "-^", ms=3, label="val (gnn-only)")
    axes[1, 0].set_title("Macro-F1"); axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3)
    axes[1, 0].legend()

    for cls in ["W", "N1", "N2", "N3", "REM"]:
        axes[1, 1].plot(epochs_x, HISTORY[f"val_f1_{cls}"], "-o", ms=3, label=cls)
    axes[1, 1].set_title("Val per-class F1"); axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3)
    axes[1, 1].legend(ncol=5, fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "training_curves.png"), dpi=80, bbox_inches="tight")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


def epoch_callback(epoch, train_loss, val_loss, train_metrics, val_metrics):
    HISTORY["epoch"].append(epoch)
    HISTORY["train_loss"].append(train_loss)
    HISTORY["val_loss"].append(val_loss)
    HISTORY["train_acc"].append(train_metrics.get("accuracy", 0.0))
    HISTORY["train_mf1"].append(train_metrics.get("macro_f1", 0.0))
    HISTORY["val_acc"].append(val_metrics.get("accuracy", 0.0))
    HISTORY["val_mf1"].append(val_metrics.get("macro_f1", 0.0))
    HISTORY["val_mf1_gnn"].append(val_metrics.get("macro_f1_gnn"))  # None ise fusion kapalı
    HISTORY["val_kappa"].append(val_metrics.get("kappa", 0.0))
    for cls in ["W", "N1", "N2", "N3", "REM"]:
        HISTORY[f"val_f1_{cls}"].append(val_metrics.get(f"f1_{cls}", 0.0))

    _print_header()
    gnn_mf1 = val_metrics.get("macro_f1_gnn")
    gnn_str = f"{gnn_mf1:>7.4f}" if gnn_mf1 is not None else "   N/A "
    row = (
        f"{epoch:>7d}",
        f"{train_loss:>7.4f}", f"{val_loss:>7.4f}",
        f"{train_metrics.get('accuracy', 0):>7.4f}",
        f"{train_metrics.get('macro_f1', 0):>7.4f}",
        f"{val_metrics.get('accuracy', 0):>7.4f}",
        f"{val_metrics.get('macro_f1', 0):>7.4f}",
        gnn_str,
        f"{val_metrics.get('kappa', 0):>7.4f}",
        f"{val_metrics.get('f1_W', 0):>7.4f}",
        f"{val_metrics.get('f1_N1', 0):>7.4f}",
        f"{val_metrics.get('f1_N2', 0):>7.4f}",
        f"{val_metrics.get('f1_N3', 0):>7.4f}",
        f"{val_metrics.get('f1_REM', 0):>7.4f}",
    )
    print(" | ".join(row))

    if wandb_run is not None:
        log = {
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/acc": train_metrics.get("accuracy", 0),
            "train/macro_f1": train_metrics.get("macro_f1", 0),
            "val/acc": val_metrics.get("accuracy", 0),
            "val/macro_f1": val_metrics.get("macro_f1", 0),
            "val/kappa": val_metrics.get("kappa", 0),
        }
        for cls in ["W", "N1", "N2", "N3", "REM"]:
            log[f"val/f1_{cls}"] = val_metrics.get(f"f1_{cls}", 0)
        # λ-fusion açıkken pure-GNN baseline'ı da logla (diagnostic)
        if gnn_mf1 is not None:
            log["val/macro_f1_gnn"] = gnn_mf1
            log["val/acc_gnn"] = val_metrics.get("accuracy_gnn", 0)
        # step=epoch + commit=True → canlı web dashboard güncellenir
        wandb_run.log(log, step=epoch, commit=True)

    _draw_live_plot()

    with open(os.path.join(OUTPUTS_DIR, "history.json"), "w") as f:
        json.dump(HISTORY, f, indent=2)


trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_dataset=train_ds,
    train_labels=data["train"]["labels"],
    val_loader=val_loader,
    config=config.train,
    data_config=config.data,
    device=device,
    spectral_extractor=spectral_ext,
    callback=epoch_callback,
)
print("Trainer hazır. Sonraki hücre eğitimi başlatır.")


In [ ]:
"""Otomatik checkpoint sürdürme ve eğitim."""
from physiographsleep.evaluation.metrics import MetricsCalculator
from physiographsleep.utils.io_utils import load_checkpoint

ckpt_path = os.path.join(config.train.checkpoint_dir, config.train.checkpoint_name)
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    print(f"Mevcut checkpoint yüklendi  : val MF1 = {ckpt['metrics']['macro_f1']:.4f}")
    print(f"Path                       : {ckpt_path}")
    print("Yeniden eğitim için bu dosyayı silip hücreyi tekrar çalıştırın.")

best_metrics = trainer.train()
print(MetricsCalculator.format_report(best_metrics))

## 6. Test Değerlendirmesi ve Post-Processing

Üç değerlendirme modu karşılaştırılır:

1. **Raw argmax** — taban çizgi, post-processing uygulanmamış softmax.
2. **Logit-bias** — val setinde Nelder–Mead ile makro-F1 maksimize eden
   toplanabilir bias vektörü (AttnSleep / SleepTransformer standartı).
3. **HMM Viterbi** — bias-düzeltilmiş logit'lerin log-softmax posteriorları
   üzerinde eğitim etiketlerinden çıkarılan geçiş matrisi ile Viterbi
   decoding (DeepSleepNet / AttnSleep standartı).

En iyi yöntem **validation** setinde seçilir; test seti yöntem seçimi
için kullanılmaz (data leakage engeli).

In [ ]:
import torch.nn.functional as F

from physiographsleep.training.evaluator import Evaluator
from physiographsleep.evaluation.metrics import MetricsCalculator
from physiographsleep.evaluation.postprocessing import HMMPostProcessor, LogitBiasOptimizer
from physiographsleep.evaluation.visualization import plot_confusion_matrix

# En iyi checkpoint'i yükle
ckpt_path = os.path.join(config.train.checkpoint_dir, config.train.checkpoint_name)
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    print(f"Yüklendi: {ckpt_path} (val MF1={ckpt['metrics']['macro_f1']:.4f})")
else:
    print("Checkpoint bulunamadı, mevcut model state kullanılıyor")

evaluator = Evaluator(device)

if "test" in data:
    test_ds = SleepEDFDataset(
        data["test"]["epochs"], data["test"]["labels"],
        config=config.data,
        spectral=data["test"].get("spectral"),
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )

    # Test ve val logits
    test_metrics, test_logits, test_labels_arr = evaluator.evaluate(
        model, test_loader, spectral_ext, return_logits=True,
    )
    val_loader_eval = DataLoader(
        val_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )
    _, val_logits, val_labels_arr = evaluator.evaluate(
        model, val_loader_eval, spectral_ext, return_logits=True,
    )

    calc = MetricsCalculator()
    results = []  # [(name, predictions, metrics), ...]

    # [1] Raw argmax — baseline
    preds_raw = test_logits.argmax(axis=1)
    raw_m = calc.compute_all(test_labels_arr, preds_raw)
    results.append(("raw", preds_raw, raw_m))

    # [2] Logit-bias (val set üzerinde Nelder-Mead ile macro-F1 maksimize):
    # AttnSleep / SleepTransformer'ın class-imbalance düzeltmesi.
    bias_opt = LogitBiasOptimizer(num_classes=5)
    bias_opt.fit(val_logits, val_labels_arr)
    preds_biased = bias_opt.apply(test_logits)
    biased_m = calc.compute_all(test_labels_arr, preds_biased)
    results.append(("biased", preds_biased, biased_m))

    # [3] HMM posterior Viterbi smoothing (DeepSleepNet/AttnSleep standart):
    # Transition matrix train labels'tan, log-softmax(bias-adjusted logits)
    # üzerinden Viterbi decoding.
    hmm = HMMPostProcessor(num_classes=5).fit(data["train"]["labels"])
    adjusted_logits = test_logits + bias_opt.bias
    log_post = F.log_softmax(torch.from_numpy(adjusted_logits), dim=-1).numpy()
    preds_hmm = hmm.smooth_posteriors(log_post)
    hmm_m = calc.compute_all(test_labels_arr, preds_hmm)
    results.append(("hmm", preds_hmm, hmm_m))

    # ── Best-method selection on VAL (not test!) ──────────────
    # Data leakage guard: post-processing method is chosen by whichever
    # method gives highest macro-F1 on the VALIDATION set. The test set
    # is never used for method selection.
    val_results = {}
    val_raw_preds = val_logits.argmax(axis=1)
    val_results["raw"] = calc.compute_all(val_labels_arr, val_raw_preds)["macro_f1"]
    val_biased_preds = bias_opt.apply(val_logits)
    val_results["biased"] = calc.compute_all(val_labels_arr, val_biased_preds)["macro_f1"]
    val_adj = val_logits + bias_opt.bias
    val_log_post = F.log_softmax(torch.from_numpy(val_adj), dim=-1).numpy()
    val_hmm_preds = hmm.smooth_posteriors(val_log_post)
    val_results["hmm"] = calc.compute_all(val_labels_arr, val_hmm_preds)["macro_f1"]
    best_pp_name = max(val_results, key=val_results.get)
    best_pp_m = {n: m for n, _, m in results}[best_pp_name]

    print("\n" + "=" * 50)
    print("TEST RESULTS")
    print("=" * 50)
    prev = None
    for name, _, m in results:
        delta = f"  ({m['macro_f1']-prev['macro_f1']:+.4f})" if prev else ""
        print(f"\n[{name}] MF1={m['macro_f1']:.4f}{delta}")
        print(MetricsCalculator.format_report(m))
        prev = m

    # Report val-selected best method
    print("\n" + "=" * 50)
    print(f"BEST METHOD (val-selected): [{best_pp_name}] "
          f"test MF1={best_pp_m['macro_f1']:.4f}")
    print(f"  Val MF1 — raw: {val_results['raw']:.4f} | "
          f"biased: {val_results['biased']:.4f} | "
          f"hmm: {val_results['hmm']:.4f}")
    print("=" * 50)
    # Warn if HMM hurts (common with FocalLoss + class imbalance)
    if val_results["hmm"] < val_results["raw"] - 0.005:
        raw_mf1 = results[0][2]["macro_f1"]
        hmm_mf1 = results[2][2]["macro_f1"]
        print(f"⚠ HMM hurts MF1 on val by {val_results['raw'] - val_results['hmm']:.4f}")
        print(f"  Test: raw N1={results[0][2].get('f1_N1',0):.4f} → "
              f"hmm N1={results[2][2].get('f1_N1',0):.4f}")

    # Confusion matrices → outputs/
    for name, preds, m in results:
        cm = MetricsCalculator.confusion_matrix(test_labels_arr, preds)
        plot_confusion_matrix(
            cm,
            save_path=os.path.join(OUTPUTS_DIR, f"cm_{name}.png"),
            title=f"{name} (MF1={m['macro_f1']:.3f})",
        )
    print(f"\nConfusion matrices kaydedildi: {OUTPUTS_DIR}")

    # Final dump
    final_metrics = {name: m for name, _, m in results}
    final_path = os.path.join(OUTPUTS_DIR, "final_model.pt")
    torch.save({
        "model": model.state_dict(),
        "config": {"num_subjects": config.data.num_subjects, "batch_size": config.data.batch_size,
                   "seq_len": config.data.seq_len},
        "metrics": final_metrics,
        "param_counts": param_counts,
    }, final_path)
    print(f"Final model: {final_path}  ({param_counts['total']:,} params)")


    if wandb_run is not None:
        for name, _, m in results:
            wandb_run.summary[f"test_{name}_mf1"] = m["macro_f1"]
            wandb_run.summary[f"test_{name}_acc"] = m["accuracy"]

            for cls in ["W", "N1", "N2", "N3", "REM"]:
                wandb_run.summary[f"test_{name}_f1_{cls}"] = m.get(f"f1_{cls}", 0)

        wandb_run.save(os.path.join(OUTPUTS_DIR, "history.json"))
        wandb_run.save(os.path.join(OUTPUTS_DIR, "*.png"))
        wandb_run.finish()

        print("WandB run kapatıldı")
    else:
        print("Test split yok")

In [ ]:
# ============================================================
# 7.05 — Calibration diagnostics (Temperature Scaling + ECE/Brier)
# ------------------------------------------------------------
# Val loss artışının "kalibrasyon kayması" mı yoksa "gerçek overfit"
# mi olduğunu ayırt eder. Temperature scaling (Guo et al. ICML 2017)
# argmax'ı korur → accuracy/F1 değişmez; sadece softmax olasılıklarını
# yumuşatır. Dolayısıyla:
#   • T ≈ 1   → zaten kalibre, val loss artışı gerçek overfit.
#   • T > 1   → over-confident (focal loss + uzun eğitim klasik).
#              NLL/ECE/Brier belirgin düşüyorsa sorun kalibrasyon.
#   • ECE yüksek ve T sonrası da yüksek kalıyorsa karar fonksiyonu
#     bozulmuş → gerçek overfit.
# ============================================================
if "test" in data:
    from physiographsleep.evaluation.postprocessing import (
        TemperatureScaling,
        compute_ece,
        compute_brier,
    )

    def _nll(logits: np.ndarray, labels: np.ndarray) -> float:
        logp = F.log_softmax(torch.from_numpy(logits).float(), dim=-1).numpy()
        return float(-logp[np.arange(len(labels)), labels].mean())

    def _softmax(logits: np.ndarray) -> np.ndarray:
        return F.softmax(torch.from_numpy(logits).float(), dim=-1).numpy()

    ts = TemperatureScaling().fit(val_logits, val_labels_arr)

    val_probs_raw  = _softmax(val_logits)
    val_probs_cal  = ts.predict_proba(val_logits)
    test_probs_raw = _softmax(test_logits)
    test_probs_cal = ts.predict_proba(test_logits)

    diag = {
        "val": {
            "nll_raw":   _nll(val_logits, val_labels_arr),
            "nll_cal":   _nll(ts.apply(val_logits), val_labels_arr),
            "ece_raw":   compute_ece(val_probs_raw,  val_labels_arr),
            "ece_cal":   compute_ece(val_probs_cal,  val_labels_arr),
            "brier_raw": compute_brier(val_probs_raw,  val_labels_arr),
            "brier_cal": compute_brier(val_probs_cal,  val_labels_arr),
        },
        "test": {
            "nll_raw":   _nll(test_logits, test_labels_arr),
            "nll_cal":   _nll(ts.apply(test_logits), test_labels_arr),
            "ece_raw":   compute_ece(test_probs_raw, test_labels_arr),
            "ece_cal":   compute_ece(test_probs_cal, test_labels_arr),
            "brier_raw": compute_brier(test_probs_raw, test_labels_arr),
            "brier_cal": compute_brier(test_probs_cal, test_labels_arr),
        },
        "T": ts.T,
    }

    print("\n" + "=" * 60)
    print("CALIBRATION DIAGNOSTICS")
    print("=" * 60)
    print(f"Fitted temperature T = {ts.T:.4f}")
    for split in ("val", "test"):
        d = diag[split]
        print(f"\n[{split}]")
        print(f"  NLL   : {d['nll_raw']:.4f} → {d['nll_cal']:.4f}  "
              f"(Δ={d['nll_cal']-d['nll_raw']:+.4f})")
        print(f"  ECE   : {d['ece_raw']:.4f} → {d['ece_cal']:.4f}  "
              f"(Δ={d['ece_cal']-d['ece_raw']:+.4f})")
        print(f"  Brier : {d['brier_raw']:.4f} → {d['brier_cal']:.4f}  "
              f"(Δ={d['brier_cal']-d['brier_raw']:+.4f})")

    # Verdict
    ece_drop = diag["val"]["ece_raw"] - diag["val"]["ece_cal"]
    nll_drop = diag["val"]["nll_raw"] - diag["val"]["nll_cal"]
    print("\nVerdict:")
    if ts.T > 1.3 and ece_drop > 0.02:
        print(f"  → Over-confident model (T={ts.T:.2f}, ECE ↓ {ece_drop:.3f}).")
        print("    Val-loss artışının büyük kısmı kalibrasyon kaymasıdır.")
    elif ts.T < 0.85:
        print(f"  → Under-confident model (T={ts.T:.2f}).")
    elif abs(ts.T - 1.0) < 0.15 and ece_drop < 0.01:
        print(f"  → Model zaten iyi kalibre (T≈1, ECE flat).")
        print("    Val-loss artışı kalibrasyonla açıklanamaz → gerçek overfit.")
    else:
        print(f"  → Karışık sinyal (T={ts.T:.2f}, ECE ↓ {ece_drop:.3f}).")

    # Persist for ablation logger
    calibration_diag = diag
else:
    calibration_diag = None


In [ ]:
# ============================================================
# 7.1 — Ablation result logger
# ------------------------------------------------------------
# Her run'ın sonucunu outputs/ablation/<ABLATION_NAME>.json dosyasına
# yazar. Ablation tablosu (en sondaki hücre) bu dosyaları okuyarak
# karşılaştırma üretir. Her toggle kombinasyonu için notebook'u baştan
# sona bir kez çalıştır → her run kendi JSON'unu kaydeder → aggregate
# hücresi hepsini basar.
# ============================================================
import json
from datetime import datetime, timezone

if "test" not in data:
    print("Test split yok, ablation log atlandı.")
else:
    ablation_dir = OUTPUTS_DIR / "ablation"
    ablation_dir.mkdir(exist_ok=True)
    out_path = ablation_dir / f"{ABLATION_NAME}.json"

    record = {
        "ablation_name": ABLATION_NAME,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "toggles": {
            "pathway":        USE_PATHWAY,
            "fusion":         USE_FUSION,
            "n1_mixup":       USE_N1_MIXUP,
            "weight_strategy": WEIGHT_STRATEGY,
        },
        "hyperparams": {
            "epochs":          config.train.epochs,
            "patience":        config.train.patience,
            "lr":              config.train.lr,
            "weight_decay":    config.train.optimizer.weight_decay,
            "label_smoothing": config.train.loss.label_smoothing,
            "n1_boost":        config.train.n1_boost,
            "warmup_epochs":   config.train.scheduler.warmup_epochs,
            "t_max":           config.train.scheduler.t_max,
            "drop_path":       config.model.graph.drop_path,
        },
        "param_counts": {k: int(v) for k, v in param_counts.items()},
        "best_val_mf1": float(ckpt["metrics"]["macro_f1"]) if os.path.exists(ckpt_path) else None,
        "test": {
            name: {
                "accuracy":    float(m["accuracy"]),
                "macro_f1":    float(m["macro_f1"]),
                "kappa":       float(m.get("kappa", 0.0)),
                "mcc":         float(m.get("mcc", 0.0)),
                "f1_W":        float(m.get("f1_W", 0.0)),
                "f1_N1":       float(m.get("f1_N1", 0.0)),
                "f1_N2":       float(m.get("f1_N2", 0.0)),
                "f1_N3":       float(m.get("f1_N3", 0.0)),
                "f1_REM":      float(m.get("f1_REM", 0.0)),
            }
            for name, _, m in results
        },
    }
    # Determine best post-processing method
    # Use val-selected method (val_results computed in test cell)
    best_pp_method = best_pp_name  # val-selected in test cell
    record["best_method"] = best_pp_method
    record["best_mf1"] = record["test"][best_pp_method]["macro_f1"]
    record["val_pp_mf1"] = {k: round(v, 4) for k, v in val_results.items()}

    # Calibration diagnostics (temperature scaling + ECE/Brier), if available
    if "calibration_diag" in globals() and calibration_diag is not None:
        record["calibration"] = {
            "T": round(calibration_diag["T"], 4),
            **{
                f"{split}_{k}": round(v, 4)
                for split in ("val", "test")
                for k, v in calibration_diag[split].items()
            },
        }

    with open(out_path, "w") as f:
        json.dump(record, f, indent=2)

    print(f"\nAblation kaydedildi: {out_path}")
    bm = best_pp_method
    bm_data = record["test"][bm]
    print(f"  Best: [{bm}] MF1 = {bm_data['macro_f1']:.4f} | "
          f"N1 F1 = {bm_data['f1_N1']:.4f} | "
          f"\u03ba = {bm_data['kappa']:.4f}")
    if bm != "hmm":
        print(f"  (HMM MF1 = {record['test']['hmm']['macro_f1']:.4f} \u2014 "
              f"worse by {bm_data['macro_f1'] - record['test']['hmm']['macro_f1']:.4f})")


## 7. Bileşen Tanılama

Eğitilmiş modelde dört ayrı ölçüm yapılır:

| Alt bölüm | Ölçüm | Başarısızlık kriteri |
|---|---|---|
| 7.1 | Aktivasyon istatistikleri (forward hook) | `std < 0.01`, `dead% > 50`, NaN/Inf |
| 7.2 | Aşama-bazlı lineer prob (waveform → graph → sequence) | Sonraki aşamada ΔMF1 ≤ 0 |
| 7.3 | Girdi modalite ablation (waveform / spectral = 0) | Tek modalite ile MF1 ≈ taban çizgisi |
| 7.4 | Füzyon tanısı (λ, branch-bazlı MF1) | σ(λ) < 0.05 veya > 0.95 |

Tüm alt hücreler yalnızca ileri geçiş yapar; parametreler güncellenmez.

In [ ]:
"""7.1 — Forward-hook aktivasyon istatistikleri.

Her ana bloğun çıkışından tek-batch μ / σ / min / max / dead% + NaN/Inf
kontrolü toplanır. Kırmızı bayraklar tabloya ek olarak yanda basılır.
"""
_stats_cache: dict[str, dict] = {}


def _tensor_stats(t: torch.Tensor) -> dict:
    t = t.detach().float()
    return {
        "shape": tuple(t.shape),
        "mean":  t.mean().item(),
        "std":   t.std().item(),
        "min":   t.min().item(),
        "max":   t.max().item(),
        "dead%": (t.abs() < 1e-6).float().mean().item() * 100.0,
        "nan":   bool(torch.isnan(t).any().item()),
        "inf":   bool(torch.isinf(t).any().item()),
    }


def _make_hook(name: str):
    def hook(_mod, _inp, out):
        t = out[0] if isinstance(out, tuple) else out
        if name not in _stats_cache and isinstance(t, torch.Tensor):
            _stats_cache[name] = _tensor_stats(t)
    return hook


_targets = ["waveform_stem", "spectral_encoder", "graph_encoder",
            "sequence_decoder", "heads"]
if model.fusion_enabled:
    _targets += ["transformer_classifier", "fusion"]

_handles = [getattr(model, n).register_forward_hook(_make_hook(n)) for n in _targets]

model.eval()
with torch.no_grad():
    _batch = next(iter(val_loader))
    _sig  = _batch["signal"].to(device)
    _spec = _batch["spectral"].to(device)
    _mask = _batch.get("mask")
    if _mask is not None:
        _mask = _mask.to(device)
    _ = model(_sig, _spec, _mask)

for _h in _handles:
    _h.remove()

print(f"{'Module':<24} {'Shape':<26} {'μ':>9} {'σ':>9} {'min':>9} {'max':>9} {'dead%':>7}")
print("-" * 96)
for name, s in _stats_cache.items():
    if s["nan"] or s["inf"]:
        flag = "  NaN/Inf"
    elif s["std"] < 0.01:
        flag = "  collapsed"
    elif s["dead%"] > 50:
        flag = f"  dead ({s['dead%']:.0f}%)"
    else:
        flag = ""
    print(
        f"{name:<24} {str(s['shape']):<26} "
        f"{s['mean']:>9.3f} {s['std']:>9.3f} "
        f"{s['min']:>9.3f} {s['max']:>9.3f} {s['dead%']:>6.1f}%{flag}"
    )

In [ ]:
"""7.2 — Aşama-bazlı lineer prob.

Her aşamanın merkez-epok temsili üzerine sınıf-dengeli
``LogisticRegression`` eğitilir (val → fit, test → eval). Art arda
aşamalarda ΔMF1 ≤ 0 ise ilgili katman diskriminatif katkı sağlamıyor
demektir.
"""
if "test_loader" not in globals():
    print("7.2 atlandı: test_loader tanımsız.")
else:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.preprocessing import StandardScaler

    _probe_val_loader = DataLoader(
        val_ds,
        batch_size=config.data.batch_size,
        shuffle=False,
        num_workers=config.data.num_workers,
        pin_memory=True,
    )

    def _extract_stage_features(model, loader):
        feats = {"raw_concat": [], "graph": [], "sequence": []}
        labels = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                sig  = batch["signal"].to(device)
                spec = batch["spectral"].to(device)
                mask = batch.get("mask")
                if mask is not None:
                    mask = mask.to(device)
                B, L, C, T = sig.shape
                center = L // 2

                flat_sig  = sig.reshape(B * L, C, T)
                flat_spec = spec.reshape(B * L, *spec.shape[2:])

                patch = model.waveform_stem(flat_sig)
                band  = model.spectral_encoder(flat_spec)
                epoch_emb = model.graph_encoder(patch, band).reshape(B, L, -1)

                P_c = patch.reshape(B, L, -1)[:, center]
                K_c = band.reshape(B, L, -1)[:, center]
                raw_concat = torch.cat([P_c, K_c], dim=-1)
                graph_c = epoch_emb[:, center]
                seq_c = model.sequence_decoder(epoch_emb, mask)[:, center]

                feats["raw_concat"].append(raw_concat.cpu().numpy())
                feats["graph"].append(graph_c.cpu().numpy())
                feats["sequence"].append(seq_c.cpu().numpy())
                labels.append(batch["label"].numpy())

        return (
            {k: np.concatenate(v, axis=0) for k, v in feats.items()},
            np.concatenate(labels),
        )

    val_feats, val_y = _extract_stage_features(model, _probe_val_loader)
    test_feats, test_y = _extract_stage_features(model, test_loader)

    print(f"{'Stage':<14} {'Dim':>8} {'MF1':>10} {'Acc':>10} {'ΔMF1':>10}")
    print("-" * 56)
    prev = None
    for stage in ("raw_concat", "graph", "sequence"):
        Xtr, Xte = val_feats[stage], test_feats[stage]
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(
            max_iter=400, n_jobs=-1, class_weight="balanced", solver="lbfgs",
        ).fit(sc.transform(Xtr), val_y)
        preds = clf.predict(sc.transform(Xte))
        mf1 = f1_score(test_y, preds, average="macro")
        acc = accuracy_score(test_y, preds)
        delta = f"{mf1 - prev:+.4f}" if prev is not None else "base"
        print(f"{stage:<14} {Xtr.shape[1]:>8d} {mf1:>10.4f} {acc:>10.4f} {delta:>10}")
        prev = mf1

In [ ]:
"""7.3 — Girdi modalite ablation.

Test setinde waveform veya spectral girdisi sıfırlanarak yeniden
değerlendirilir; parametreler sabittir. Her branch'in göreli katkısı
MF1 düşüşü ile ölçülür. Waveform branch'te `Conv1d(bias=False)` + GN,
spectral branch'te `Linear(bias=True)` olduğundan asimetrik davranış
beklenir.
"""
if "test_loader" not in globals():
    print("7.3 atlandı: test_loader tanımsız.")
else:
    from sklearn.metrics import accuracy_score, f1_score

    def _eval_modality(zero_spec: bool = False, zero_wave: bool = False):
        preds, labels = [], []
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                sig  = batch["signal"].to(device)
                spec = batch["spectral"].to(device)
                if zero_spec:
                    spec = torch.zeros_like(spec)
                if zero_wave:
                    sig = torch.zeros_like(sig)
                mask = batch.get("mask")
                if mask is not None:
                    mask = mask.to(device)
                out = model(sig, spec, mask)
                preds.append(out["stage"].argmax(dim=-1).cpu().numpy())
                labels.append(batch["label"].numpy())
        y = np.concatenate(labels)
        p = np.concatenate(preds)
        return f1_score(y, p, average="macro"), accuracy_score(y, p)

    print(f"{'Condition':<28} {'MF1':>8} {'ΔMF1':>9} {'Acc':>8}")
    print("-" * 56)
    base_mf1, base_acc = _eval_modality()
    print(f"{'Baseline (both)':<28} {base_mf1:>8.4f} {'base':>9} {base_acc:>8.4f}")
    for label, zs, zw in (
        ("Only waveform (spec=0)", True,  False),
        ("Only spectral (sig=0)",  False, True),
        ("Both zero (sanity)",     True,  True),
    ):
        mf1, acc = _eval_modality(zero_spec=zs, zero_wave=zw)
        print(f"{label:<28} {mf1:>8.4f} {mf1 - base_mf1:>+9.4f} {acc:>8.4f}")

In [ ]:
"""7.4 — Füzyon tanısı (λ + branch-bazlı MF1).

σ(λ) ≈ 0 ise transformer branch etkisiz, ≈ 1 ise GNN branch ihmal
edilmiş demektir. 0.2–0.8 aralığı gerçek ensemble davranışına karşılık
gelir.
"""
if "test_loader" not in globals():
    print("7.4 atlandı: test_loader tanımsız.")
elif not getattr(model, "fusion_enabled", False):
    print("7.4 atlandı: fusion disabled.")
else:
    from sklearn.metrics import accuracy_score, f1_score

    with torch.no_grad():
        lam = model.fusion.lambda_value.item()
    print(f"σ(λ) = {lam:.4f}  "
          f"(transformer={lam*100:.1f}%, gnn={(1-lam)*100:.1f}%)")
    if lam < 0.05 or lam > 0.95:
        print("  uyarı: füzyon dejenere — tek branch baskın.")

    fused, gnn, trans, labels = [], [], [], []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            sig  = batch["signal"].to(device)
            spec = batch["spectral"].to(device)
            mask = batch.get("mask")
            if mask is not None:
                mask = mask.to(device)
            out = model(sig, spec, mask)
            fused.append(out["stage"].argmax(dim=-1).cpu().numpy())
            gnn.append(out["stage_gnn"].argmax(dim=-1).cpu().numpy())
            trans.append(out["stage_transformer"].argmax(dim=-1).cpu().numpy())
            labels.append(batch["label"].numpy())
    y = np.concatenate(labels)

    print(f"\n{'Branch':<18} {'MF1':>8} {'Acc':>8} {'F1_N1':>8}")
    print("-" * 46)
    for name, arr in (
        ("fused (output)", np.concatenate(fused)),
        ("gnn-only",       np.concatenate(gnn)),
        ("trans-only",     np.concatenate(trans)),
    ):
        mf1 = f1_score(y, arr, average="macro")
        acc = accuracy_score(y, arr)
        f1n1 = f1_score(y, arr, average=None, labels=[0, 1, 2, 3, 4])[1]
        print(f"{name:<18} {mf1:>8.4f} {acc:>8.4f} {f1n1:>8.4f}")

## 8. Ablation Toplu Tablosu

`outputs/ablation/*.json` dosyalarının makro-F1'e göre sıralı
karşılıklı karşılaştırması. `USE_PATHWAY / USE_FUSION / USE_N1_MIXUP`
kombinasyonlarının etkisini tek tabloda ve makale için Markdown
formatında raporlar.

In [ ]:
# ============================================================
# Ablation aggregate table
# ------------------------------------------------------------
# outputs/ablation/*.json dosyalarını okur, HMM-post-processing
# MF1'e göre sıralar. Her run için toggle combo, test MF1/ACC/κ ve
# per-class F1'leri tek tabloda gösterir.
# ============================================================
import json
from pathlib import Path

ablation_dir = OUTPUTS_DIR / "ablation"
if not ablation_dir.exists():
    print(f"Henüz kayıt yok: {ablation_dir}")
else:
    records = []
    for p in sorted(ablation_dir.glob("*.json")):
        try:
            records.append(json.loads(p.read_text()))
        except Exception as e:
            print(f"[skip] {p.name}: {e}")

    if not records:
        print(f"Henüz kayıt yok: {ablation_dir}")
    else:
        # Best MF1'e göre sırala (her run'ın en iyi post-processing yöntemi)
        def _best_mf1(r):
            return max(v["macro_f1"] for v in r["test"].values())
        records.sort(key=_best_mf1, reverse=True)

        print(f"\n{'Run':<40s} {'P':>2s} {'F':>2s} {'M':>2s} "
              f"{'Raw MF1':>8s} {'Bias MF1':>9s} {'HMM MF1':>8s} "
              f"{'Best':>5s} {'BestMF1':>8s} {'Acc':>8s} {'κ':>7s} {'F1_N1':>7s} {'F1_N2':>7s}")
        print("-" * 140)
        for r in records:
            t = r["toggles"]
            raw = r["test"]["raw"]
            bia = r["test"]["biased"]
            hmm = r["test"]["hmm"]
            bm_name = r.get("best_method", max(r["test"], key=lambda k: r["test"][k]["macro_f1"]))
            bm = r["test"][bm_name]
            print(
                f"{r['ablation_name']:<40s} "
                f"{'1' if t['pathway'] else '0':>2s} "
                f"{'1' if t['fusion']   else '0':>2s} "
                f"{'1' if t['n1_mixup'] else '0':>2s} "
                f"{raw['macro_f1']:>8.4f} {bia['macro_f1']:>9.4f} {hmm['macro_f1']:>8.4f} "
                f"{bm_name:>5s} {bm['macro_f1']:>8.4f} "
                f"{bm['accuracy']:>8.4f} {bm['kappa']:>7.4f} "
                f"{bm['f1_N1']:>7.4f} {bm['f1_N2']:>7.4f}"
            )

        # ΔMF1 analizi: baseline (hepsi OFF) varsa her toggle'ın izole
        # katkısını göster.
        def find_combo(pw: bool, fu: bool, mx: bool):
            for r in records:
                t = r["toggles"]
                if t["pathway"] == pw and t["fusion"] == fu and t["n1_mixup"] == mx:
                    return r
            return None

        base = find_combo(False, False, False)
        if base is not None:
            base_best = _best_mf1(base)
            base_bm = base.get("best_method", max(base["test"], key=lambda k: base["test"][k]["macro_f1"]))
            print(f"\nBaseline (all OFF) Best MF1 = {base_best:.4f} [{base_bm}]")
            for name, pw, fu, mx in [
                ("+pathway only",  True,  False, False),
                ("+fusion only",   False, True,  False),
                ("+mixup only",    False, False, True),
                ("+path+fusion",   True,  True,  False),
                ("+path+mixup",    True,  False, True),
                ("+fusion+mixup",  False, True,  True),
                ("all ON",         True,  True,  True),
            ]:
                r = find_combo(pw, fu, mx)
                if r is None:
                    continue
                r_best = _best_mf1(r)
                r_bm = r.get("best_method", max(r["test"], key=lambda k: r["test"][k]["macro_f1"]))
                dm = r_best - base_best
                print(f"  {name:<18s}: {r_best:.4f} [{r_bm}]  "
                      f"(Δ{dm:+.4f})")

        # Markdown tablosu (makale için)
        print("\n\n### Markdown table\n")
        print("| Run | Pathway | Fusion | Mixup | Raw MF1 | Biased MF1 | HMM MF1 | Best | Best MF1 | Acc | \u03ba | F1_N1 |")
        print("|---|:-:|:-:|:-:|---:|---:|---:|:---:|---:|---:|---:|---:|")
        for r in records:
            t = r["toggles"]
            bm_name = r.get("best_method", max(r["test"], key=lambda k: r["test"][k]["macro_f1"]))
            bm = r["test"][bm_name]
            print(
                f"| {r['ablation_name']} "
                f"| {'\u2713' if t['pathway'] else '\u2013'} "
                f"| {'\u2713' if t['fusion']   else '\u2013'} "
                f"| {'\u2713' if t['n1_mixup'] else '\u2013'} "
                f"| {r['test']['raw']['macro_f1']:.4f} "
                f"| {r['test']['biased']['macro_f1']:.4f} "
                f"| {r['test']['hmm']['macro_f1']:.4f} "
                f"| {bm_name} "
                f"| **{bm['macro_f1']:.4f}** "
                f"| {bm['accuracy']:.4f} "
                f"| {bm['kappa']:.4f} "
                f"| {bm['f1_N1']:.4f} |"
            )


## 9. K-Fold Subject-Wise Cross-Validation

Literatürle adil karşılaştırma için zorunlu olan protokol.
`physiographsleep.scripts.run_cv` CLI'si kullanılır; her fold bir test
öznesi (veya EDF-78 için 5-fold) üzerinde eğitim-tekrarı yapar ve
sonuçları JSONL dosyasına yazar.

| Veri seti | Önerilen protokol | Yaklaşık süre (T4) | Hedef MF1 |
|---|---|---|---|
| Sleep-EDF-20 | 20-fold LOSO | ~4 saat | 0.77 – 0.80 |
| Sleep-EDF-78 | 5-fold subject-wise | ~20–25 saat | 0.78 – 0.80 |

Fold sonuçları `outputs/cv_edf{20,78}.jsonl` dosyalarına eklenir;
son hücre mean ± std ile literatür hedef bandına karşı verdikt verir.

In [ ]:
"""9.1 — Sleep-EDF-20 leave-one-subject-out (20-fold).

Cache zaten mevcutsa (`__persubj__` npz), yeniden indirme yapılmaz.
Çıktı: fold başına bir JSON satırı + özet satırı.
"""
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "physiographsleep.scripts.run_cv",
        "--num-subjects", "20",
        "--folds", "20",
        "--epochs", str(config.train.epochs),
        "--batch-size", str(config.data.batch_size),
        "--log-dir", str(PROJECT_DIR / "logs" / "cv_edf20"),
        "--out", str(OUTPUTS_DIR / "cv_edf20.jsonl"),
    ],
    check=False,
)

In [ ]:
"""9.2 — Sleep-EDF-78 subject-wise 5-fold.

İlk çağrı tam `sleep-cassette` setini (~2.5 GB) indirir ve
`sleepedf78_persubj_*.npz` önbelleğini oluşturur. Tahmini süre: ~5
saat/fold × 5 fold ≈ 25 saat (T4). Hızlı doğrulama için `--epochs 15`
(erken durma etkin).
"""
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "physiographsleep.scripts.run_cv",
        "--num-subjects", "78",
        "--folds", "5",
        "--epochs", str(config.train.epochs),
        "--batch-size", str(config.data.batch_size),
        "--log-dir", str(PROJECT_DIR / "logs" / "cv_edf78"),
        "--out", str(OUTPUTS_DIR / "cv_edf78.jsonl"),
    ],
    check=False,
)

In [ ]:
"""9.3 — CV sonuçlarının özeti ve literatür karşılaştırması."""
import json
from pathlib import Path


def summarize_cv(jsonl_path: Path | str, label: str,
                 target: tuple[float, float]) -> None:
    path = Path(jsonl_path)
    if not path.exists():
        print(f"[{label}] dosya bulunamadı: {path}")
        return
    folds = [
        json.loads(l) for l in path.read_text().splitlines()
        if l.strip() and "fold" in json.loads(l)
    ]
    if not folds:
        print(f"[{label}] fold kaydı yok.")
        return

    mf1   = np.array([r["test_macro_f1"] for r in folds])
    acc   = np.array([r["test_accuracy"] for r in folds])
    kappa = np.array([r.get("test_kappa", 0.0) for r in folds])
    n1 = np.array([
        (r["test_per_class_f1"][1] if len(r.get("test_per_class_f1", [])) >= 2
         else np.nan)
        for r in folds
    ])

    print(f"\n{label}  ({len(folds)} folds)")
    print(f"{'fold':<6}{'test_subjects':<22}{'MF1':>8}{'Acc':>8}{'κ':>8}{'F1_N1':>8}")
    for r in folds:
        subs = ",".join(r.get("test_subjects", []))[:20]
        pc = r.get("test_per_class_f1", [])
        n1_val = pc[1] if len(pc) >= 2 else float("nan")
        print(f"{r['fold']:<6}{subs:<22}"
              f"{r['test_macro_f1']:>8.4f}"
              f"{r['test_accuracy']:>8.4f}"
              f"{r.get('test_kappa', 0.0):>8.4f}"
              f"{n1_val:>8.4f}")

    def _stat(arr):
        return arr.mean(), arr.std(ddof=1) if arr.size > 1 else 0.0

    mf1_m, mf1_s = _stat(mf1)
    acc_m, acc_s = _stat(acc)
    kap_m, kap_s = _stat(kappa)

    lo, hi = target
    verdict = (
        "hedef bandında" if lo <= mf1_m <= hi
        else ("hedef altında" if mf1_m < lo else "hedef üstünde")
    )
    print(f"\nMF1   = {mf1_m:.4f} ± {mf1_s:.4f}   (target {lo:.2f}–{hi:.2f}, {verdict})")
    print(f"Acc   = {acc_m:.4f} ± {acc_s:.4f}")
    print(f"κ     = {kap_m:.4f} ± {kap_s:.4f}")
    if not np.isnan(n1).all():
        n1_m = np.nanmean(n1)
        n1_s = np.nanstd(n1, ddof=1) if (~np.isnan(n1)).sum() > 1 else 0.0
        print(f"N1    = {n1_m:.4f} ± {n1_s:.4f}")


summarize_cv(OUTPUTS_DIR / "cv_edf20.jsonl", "Sleep-EDF-20 LOSO", (0.77, 0.80))
summarize_cv(OUTPUTS_DIR / "cv_edf78.jsonl", "Sleep-EDF-78 5-fold", (0.78, 0.80))